In [ ]:
import time
import board
from adafruit_bno08x.i2c import BNO08X_I2C
import busio
from adafruit_bno08x import (BNO_REPORT_MAGNETOMETER,BNO_REPORT_GEOMAGNETIC_ROTATION_VECTOR)

i2c = busio.I2C(board.SCL, board.SDA, frequency=800000)
bno = BNO08X_I2C(i2c)
bno.enable_feature(BNO_REPORT_MAGNETOMETER)
bno.enable_feature(BNO_REPORT_GEOMAGNETIC_ROTATION_VECTOR)

print("✅ BNO085 Initialized!")
print("Reading sensor data...")
while True:
    try:
    
        mag_x, mag_y, mag_z = bno.magnetic
        geo_quater= bno.geomagnetic_quaternion

        print("X: %0.6f  Y: %0.6f Z: %0.6f uT" % (mag_x, mag_y, mag_z))
        print("Geo X: %0.6f  Y: %0.6f Z: %0.6f" % (geo_quater[0], geo_quater[1], geo_quater[2]))
        print("")
        time.sleep(0.1)
       
    except RuntimeError as e:
        # 3. HANDLE GLITCHES
        # If we get "Unprocessable Batch bytes", just skip this loop iteration
        # print(f"Glitch ignored: {e}") # Uncomment to see how often it happens
        continue
    except KeyError as ke:
        # 3. HANDLE GLITCHES
        # If we get "Unprocessable Batch bytes", just skip this loop iteration
        # print(f"Glitch ignored: {e}") # Uncomment to see how often it happens
        continue
    except OSError as oe:
        # 3. HANDLE GLITCHES
        # If we get "Unprocessable Batch bytes", just skip this loop iteration
        # print(f"Glitch ignored: {e}") # Uncomment to see how often it happens
        continue
    except KeyboardInterrupt:
        print("\n\n✅ Stopped reading.")


In [ ]:
#!/usr/bin/env python3
"""
BNO085 Compass - Shows heading relative to North
"""

import time
import math
import board
import busio
from adafruit_bno08x import (
    BNO_REPORT_ROTATION_VECTOR,
   # BNO_REPORT_GEOMAGNETIC_ROTATION_VECTOR
)
from adafruit_bno08x.i2c import BNO08X_I2C

def quaternion_to_heading(quat):
    """Convert quaternion to compass heading in degrees"""
    i, j, k, real = quat
    
    heading = math.atan2(
        2.0 * (real * k + i * j),
        1.0 - 2.0 * (j * j + k * k)
    )
    
    heading = math.degrees(heading)
    if heading < 0:
        heading += 360
    
    return heading

def main():
    print("Initializing BNO085...")
    # Initialize I2C with slower speed for stability
    i2c = busio.I2C(board.SCL, board.SDA, frequency=100000)
    time.sleep(0.1)
    
    # Try both possible addresses
    try:
        bno = BNO08X_I2C(i2c, address=0x4A)
        print("Found BNO085 at 0x4A")
    except:
        bno = BNO08X_I2C(i2c, address=0x4B)
        print("Found BNO085 at 0x4B")
    '''
    # Reset the sensor
    print("Resetting sensor...")
    bno.soft_reset()
    time.sleep(1)
    '''
    # Enable geomagnetic rotation vector (better for compass)
    print("Enabling compass mode...")
    bno.enable_feature(BNO_REPORT_ROTATION_VECTOR)
    time.sleep(1)
    
    print("\nCompass Ready!")
    print("=" * 50)
    print("North=0° | East=90° | South=180° | West=270°\n")
    while True:
        try:
            quat = bno.geomagnetic_quaternion
            
            heading = quaternion_to_heading(quat)
            
            # Cardinal direction
            dirs = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
            direction = dirs[round(heading / 45) % 8]
            
            print(f"\r{heading:6.1f}° {direction:3s}", 
                    end='', flush=True)
            
            time.sleep(0.4)
        

        except Exception as e:
            print(f"\nError: {e}")
            print("\nTroubleshooting:")
            print("1. Check wiring (SDA, SCL, VCC, GND)")
            print("2. Verify I2C is enabled: sudo raspi-config")
            print("3. Check I2C devices: i2cdetect -y 1")
            continue

        except KeyboardInterrupt:
            print("\n\nStopped")
            
if __name__ == "__main__":
    main()

Initializing BNO085...
Found BNO085 at 0x4A
Enabling compass mode...


RuntimeError: ('Was not able to enable feature', 9)

In [ ]:
import time
import board
import busio
from adafruit_bno08x.i2c import BNO08X_I2C
from adafruit_bno08x import (
    BNO_REPORT_MAGNETOMETER,
    BNO_REPORT_ACCELEROMETER,
    BNO_REPORT_GYROSCOPE,
    BNO_REPORT_GEOMAGNETIC_ROTATION_VECTOR,
)

print("Initializing BNO085...")

# I2C setup
i2c = busio.I2C(board.SCL, board.SDA)
bno = BNO08X_I2C(i2c)

# Reset sensor
bno.soft_reset()
time.sleep(1)

# Enable required reports
bno.enable_feature(BNO_REPORT_MAGNETOMETER)
bno.enable_feature(BNO_REPORT_ACCELEROMETER)
bno.enable_feature(BNO_REPORT_GYROSCOPE)
bno.enable_feature(BNO_REPORT_GEOMAGNETIC_ROTATION_VECTOR)

time.sleep(1)

print("\n=== BNO085 CALIBRATION MODE ===")
print("Follow the steps below:")
print("1) Rotate flat in a circle")
print("2) Tilt 45° and rotate")
print("3) Figure-8 motion")
print("4) Hold still for gyro\n")

start_time = time.time()

while True:
    try:
        mag = bno.magnetic
        acc = bno.acceleration
        gyr = bno.gyro
        quat = bno.geomagnetic_quaternion
        print(
            f"\rMAG {mag} | ACC {acc} | GYR {gyr}",
            end="",
            flush=True
        )

        # After 60 seconds, calibration is usually complete
        if time.time() - start_time > 60:
            print("\n\n✅ Calibration time completed")
            print("You may now stop the script")
            break

        time.sleep(0.2)
    except RuntimeError as e:
        # 3. HANDLE GLITCHES
        # If we get "Unprocessable Batch bytes", just skip this loop iteration
        # print(f"Glitch ignored: {e}") # Uncomment to see how often it happens
        continue
    except KeyError as ke:
        # 3. HANDLE GLITCHES
        # If we get "Unprocessable Batch bytes", just skip this loop iteration
        # print(f"Glitch ignored: {e}") # Uncomment to see how often it happens
        continue
    except OSError as oe:
        # 3. HANDLE GLITCHES
        # If we get "Unprocessable Batch bytes", just skip this loop iteration
        # print(f"Glitch ignored: {e}") # Uncomment to see how often it happens
        continue
    except KeyboardInterrupt:
        print("\n\n✅ Stopped reading.")

In [ ]:
import time
import board
import busio
from adafruit_bno08x.i2c import BNO08X_I2C
from adafruit_bno08x import (
    BNO_REPORT_ROTATION_VECTOR,
    BNO_REPORT_GAME_ROTATION_VECTOR,
)

i2c = busio.I2C(board.SCL, board.SDA, frequency=400000)
bno = BNO08X_I2C(i2c)

print("Soft reset...")
bno.soft_reset()

# Enable BOTH reports (this matters)
bno.enable_feature(BNO_REPORT_GAME_ROTATION_VECTOR, 10000)
bno.enable_feature(BNO_REPORT_ROTATION_VECTOR, 10000)

print("Move the sensor slowly in all directions")
print("Accuracy: 0=Unreliable, 1=Low, 2=Medium, 3=High\n")

start = time.monotonic()

while True:
    try:
        quat = bno.quaternion
        game_quat = bno.game_quaternion
        acc = bno.calibration_status

        print(
            f"Acc: {acc} | "
            f"Quat: {quat} | "
            f"GameQuat: {game_quat}"
        )

        if acc == 3:
            print("✅ FULLY CALIBRATED")
            break

        if time.monotonic() - start > 180:
            print("⏱️ Timeout (3 minutes)")
            break

    except RuntimeError:
        # Happens occasionally if data not ready
        continue
    except KeyError as ke:
        # 3. HANDLE GLITCHES
        # If we get "Unprocessable Batch bytes", just skip this loop iteration
        # print(f"Glitch ignored: {e}") # Uncomment to see how often it happens
        continue
    except OSError as oe:
        # 3. HANDLE GLITCHES
        # If we get "Unprocessable Batch bytes", just skip this loop iteration
        # print(f"Glitch ignored: {e}") # Uncomment to see how often it happens
        continue
    except IndexError as ie:
        # 3. HANDLE GLITCHES
        # If we get "Unprocessable Batch bytes", just skip this loop iteration
        # print(f"Glitch ignored: {e}") # Uncomment to see how often it happens
        continue 
    except KeyboardInterrupt:
        print("\n\n✅ Stopped reading.")

    time.sleep(0.1)

Soft reset...
Move the sensor slowly in all directions
Accuracy: 0=Unreliable, 1=Low, 2=Medium, 3=High

Acc: 0 | Quat: (-0.02642822265625, 0.046630859375, 0.9505615234375, 0.305908203125) | GameQuat: (0.03369140625, 0.04144287109375, 0.00244140625, 0.99859619140625)
Acc: 0 | Quat: (-0.02716064453125, 0.03936767578125, 0.956298828125, 0.28851318359375) | GameQuat: (0.02886962890625, 0.03717041015625, 0.004150390625, 0.9989013671875)
Acc: 0 | Quat: (-0.0255126953125, 0.0408935546875, 0.958740234375, 0.2802734375) | GameQuat: (0.031494140625, 0.03570556640625, 0.00885009765625, 0.99884033203125)
Acc: 0 | Quat: (-0.14288330078125, 0.15216064453125, -0.97607421875, 0.0614013671875) | GameQuat: (-0.10614013671875, -0.19708251953125, 0.31201171875, 0.92333984375)
Acc: 0 | Quat: (-0.05609130859375, 0.0345458984375, -0.629638671875, 0.7740478515625) | GameQuat: (-0.02593994140625, -0.06610107421875, 0.9044189453125, 0.42071533203125)

		********** Packet *************
DBG::		 HEADER:
DBG::		 Da

KeyboardInterrupt: 